# Blatt 09, Aufgabe 20

In [ ]:
# installing modules, kann übersprungen werden, weil es nur selten funktioniert wie gedacht
#%pip install torch
#%pip install mace-torch
#%pip install ase
#%pip install nglview
#%pip install ipywidgets

## Modulimport und Definitionen

Die aktuelle Situation macht die Nutzung klassischer Quantenchemischer Simulationssoftware (hier: Gaussian) nicht möglich. Alternativ wird MACE genutzt, welches auf _Machine Learning_ basiert. Das heißt, statt explizit quantenchemische Gleichungen zu lösen, wird über einen Datensatz unterschiedlicher Art und Größe interpoliert.

In der heutigen Aufgabe Blatt09-20 wollen wir Ergebnisse aus verschiedenen Theorieniveaus miteinander vergleichen:
1. GGA: PBE (_MACE-matpes-pbe-omat-ft.model_)
2. METAGGA: R2SCAN (_MACE-matpes-r2scan-omat-ft.model_)
3. Hybrid (mit Dispersionskorrektur): wB97M+D3 (_MACE-OFF24_medium.model_)
4. Coupled Cluster (_mace_anicc_), als Alternative PBE+U (_mace_mp(device='cpu')_)

Typischerweise würde man mit steigender Komplexität deutlich höheren Rechenaufwand und eine bessere Genauigkeit erwarten. Wir werden uns das am Beispiel von H<sub>2</sub> anschauen. Dafür werden wir die Struktur mit jedem der Modelle optimieren und folgende Eigenschaften mitenander vergleichen:
1. Den H-H-Gleichgewichtsabstand
2. Die H<sub>2</sub>-Gesamtenergie
3. Die Bindungsenergie E<sub>H<sub>2</sub></sub><sup>bind</sup> = E<sub>H<sub>2</sub></sub><sup>ges</sup> - 2 E<sub>H</sub><sup>ges</sup>

Erstellen wir zuerst eine Strukturdatei als _.xyz_. In einer xyz Datei wird nur die chemische Spezie und ihre Position in der Struktur angegeben, als Beispiel für H<sub>2</sub>O:

3 # erste Zeile mit Anzahl der Atomen in der Struktur

Water molecule (H2O) # Kommentarzeile

O       0.000000    0.000000    0.000000 # Spezie und Position von Atom 1

H       0.757107    0.000000    0.587006 # Spezie und Position von Atom 2

H       0.757107    0.000000   -0.587006 # Spezie und Position von Atom 3


Erstellen Sie nun für H<sub>2</sub> und H eine Startstruktur mit den Namen "H2.xyz" und "H.xyz". Anschließend wollen wir unsere verschiedenen Module importieren und definieren, dabei sind bis auf das _mace_anicc_-Modell alle Modelle bereits als Datei gegeben:

In [ ]:
# MACE um die Rechner und Modelle zu definieren
from mace.calculators import mace_anicc # coupled cluster model
from mace.calculators import mace_mp # standard model
from mace.calculators import MACECalculator # Calculator um Rechner zu definieren

# ASE um Strukturen als atoms object zu lesen, schreiben und ändern
from ase.io import read, write # Zum Lesen und Schreiben von Strukturdateien
from ase.visualize import view # Zur Visualisierung von Strukturen
from ase.constraints import ExpCellFilter # Wird für die Strukturoptimierung von periodischen Systemen benötigt
from ase.optimize import BFGS # Solver für die Berechnungen

# nur für Utility
import copy # Objekt muss kopiert werden, sonst wird es verändert
import time # Um Zeit für den Funktionsdurchlauf zu berechnen
import numpy as np # Utility für Vektoren

Nachdem wir die Module importiert haben und nutzen können, müssen wir uns Gedanken darüber machen, welche Eigenschaften wir berechnen müssen/wollen:
1. Gleichgewichtsabstand: Länge des Vektors zwischen einem H zum anderen H.
2. H<sub>2</sub>-Gesamtenergie E<sub>H<sub>2</sub></sub><sup>ges</sup>: Wird von ASE über den mit MACE definierten _calculator_ als object.get_total_energy() berechnet (_object_ ist hier die Strukturdatei, Beispiel siehe später).
3. E<sub>H<sub>2</sub></sub><sup>bind</sup> = E<sub>H<sub>2</sub></sub><sup>ges</sup> - 2 E<sub>H</sub><sup>ges</sup> : Energiedifferenz von H<sub>2</sub> zu zwei isolierten H.

Wir müssen also drei Funktionen definieren und Rechnungen für zwei Strukturen (H<sub>2</sub> und isoliertes H) durchführen. Dafür definieren wir zuerst eine Funktion für die Strukturoptimierung, eine Funktion zur Energieberechnung ist bereits in ASE enthalten. Die Ergebnisse können Sie in die Ergebnistabelle ganz am Ende eintragen. Definieren wir zuerst die Funktion zur Strukturoptimierung:

In [ ]:
def relaxation_mace(atoms, calc, fix_cell = True, f = 0.01):
    '''
    Parameters
    ----------
    structure : ATOMS object
        Struktur die für die Optimierung genutzt werden soll.
    calc : CALCULATOR object
        Rechner der im weiteren Laufe zur Simulation genutzt werden soll.
    fix_cell : BOOL, optional
        Beschreibt, ob die Gitterkonstanten fix oder veränderlich sind. The default is True.
    f : FLOAT, optional
        Konvergenzkriterium zum Beenden der Optimierung. The default is 0.01.

    Returns
    -------
    structure : ATOMS object
        Beinhaltet die optimierte Struktur als ATOMS object.
    '''
    # define calculator
    t_start = time.time()
    structure = copy.deepcopy(atoms)
    structure.calc = calc # Hier wird ein zusätzlicher Rechner definiert, auf Grundlage dessen alle weiteren Rechnungen durchgeführt werden
    
    # check for constraints and run
    if fix_cell == True: # kleine Anpassung für periodische Systeme, da wird ein ExpCellFilter gebraucht
        cell_filter = ExpCellFilter(structure)
        opt = BFGS(cell_filter) # BFGS ist ein Solver
    else:    
        opt = BFGS(structure)
        opt.run(fmax=f)  # Optimize until forces < 0.01 eV/Å   
        pass
    
    print("Optimizing geometry...")
    opt.run(fmax=f)  # Optimise until forces < 0.01 eV/Å
    # set up dict with relaxed structure
    t_end = time.time()
    print(f'\nDie Funktion brauchte {t_end-t_start} Sekunden.')
    return structure

Zur Berechnung der Abstände definieren wir vorerst keine Funktion, das können wir am Ende zügig rechnen. Die _calculator_ müssen wir jedoch defineren:

In [ ]:
calc_gga = MACECalculator(model_paths="./MACE-matpes-pbe-omat-ft.model",device='cpu')
calc_metagga = MACECalculator(model_paths="./MACE-matpes-r2scan-omat-ft.model",device='cpu')
calc_hybrid = MACECalculator(model_paths='./MACE-OFF24_medium.model',device='cpu') # medium heißt hier, das Modell ist kleiner als der Standard
calc_cc = mace_mp(default_dtype='float64',device='cpu') #mace_anicc(device='cpu')

Nun können wir unsere weiter oben gefertigten Strukturen importieren:

In [ ]:
H2 = read('./H2.xyz',format='xyz')
H = read('./H.xyz',format='xyz')

print(f"Der Atomabstand in der Input-Struktur ist {np.linalg.norm(H2.get_positions()[1]-H2.get_positions()[0]):.2f} Angstrom.\n")
view(H2,viewer='ngl')

## Strukturoptimierung und Atomabstand

Nachdem beide Strukturen erstellt und importiert sind, können wir eine Strukturoptimierung vornehmen, die Energien berechnen lassen und in unsere Ergebnistabelle eintragen:

In [ ]:
# coupled cluster
h2_cc = relaxation_mace(H2, calc_cc, fix_cell = False, f = 0.01)
h_cc = relaxation_mace(H, calc_cc, fix_cell = False, f = 0.01)

# hybrid
h2_hybrid = relaxation_mace(H2, calc_hybrid, fix_cell = False, f = 0.01)
h_hybrid = relaxation_mace(H, calc_hybrid, fix_cell = False, f = 0.01)

# metagga
h2_metagga = relaxation_mace(H2, calc_metagga, fix_cell = False, f = 0.01)
h_metagga = relaxation_mace(H, calc_metagga, fix_cell = False, f = 0.01)

# gga
h2_gga = relaxation_mace(H2, calc_gga, fix_cell = False, f = 0.01)
h_gga = relaxation_mace(H, calc_gga, fix_cell = False, f = 0.01)

# print alle Atomabstände
print(f"Der Atomabstand im Coupled Cluster Modell ist {np.linalg.norm(h2_cc.get_positions()[1]-h2_cc.get_positions()[0]):.2f} Angstrom.\n")
print(f"Der Atomabstand im Hybrid Modell ist {np.linalg.norm(h2_hybrid.get_positions()[1]-h2_hybrid.get_positions()[0]):.2f} Angstrom.\n")
print(f"Der Atomabstand im METAGGA Modell ist {np.linalg.norm(h2_metagga.get_positions()[1]-h2_metagga.get_positions()[0]):.2f} Angstrom.\n")
print(f"Der Atomabstand im GGA Modell ist {np.linalg.norm(h2_gga.get_positions()[1]-h2_gga.get_positions()[0]):.2f} Angstrom.\n")

view(h2_metagga,viewer='ngl')

Der experimentelle Bindungsabstand in Wasserstoff beträgt ungefähr 0.74 Angström.

## Energieberechnungen

Nun, da wir die Bindungsabstände haben, können wir uns der Berechnung der Energien widmen. Wir haben bereits alle Strukturen optimiert und können ihre Energien mit _.get_total_energy()_ erhalten. Im folgenden berechnen wir nun die Bindungsenergien:

In [ ]:
# Coupled Cluster
Ebind_cc = h2_cc.get_total_energy() - 2*h_cc.get_total_energy()
print(f" Die Bindungsenergie im Coupled Cluster Modell ist {Ebind_cc:.2f} eV oder {Ebind_cc*96.485:2f} kJ/mol.")
# Hybrid
Ebind_hybrid = h2_hybrid.get_total_energy() - 2*h_hybrid.get_total_energy()
print(f" Die Bindungsenergie im Hybrid Modell ist {Ebind_hybrid:.2f} eV oder {Ebind_hybrid*96.485:2f} kJ/mol.")

# METAGGA
Ebind_metagga = h2_metagga.get_total_energy() - 2*h_metagga.get_total_energy()
print(f" Die Bindungsenergie im METAGGA Modell ist {Ebind_metagga:.2f} eV oder {Ebind_metagga*96.485:2f} kJ/mol.")

# GGA
Ebind_gga = h2_gga.get_total_energy() - 2*h_gga.get_total_energy()
print(f" Die Bindungsenergie im GGA Modell ist {Ebind_gga:.2f} eV oder {Ebind_gga*96.485:2f} kJ/mol.")

Die experimentell gemessene Bindungsenergie beträgt - 436 kJ/mol. In vielen Fällen wird die Bindungsenergie nicht als Energiegewinn durch Bindung beschrieben, sondern als die Energie, die aufgewendet werden muss, um die Bindung zu brechen. Während im ersten Fall eine negativere Energie auf eine stärkere Bindung hinweist, ist es im letzten Fall umgekehrt: Je positiver die Energie, desto stärker die Bindung.

## Ergebnistabelle

 | Funktional/Modell | H<sub>2</sub>-Gleichgewichtsabstand (A) | E<sub>H<sub>2</sub></sub><sup>ges</sup> (eV) | E<sub>H</sub><sup>ges</sup> | E<sub>H<sub>2</sub></sub> | <sup>bind</sup> (eV) | 
 | :--------: | :--------: | :--------: | :--------: | :--------: | 
 | PBE (GGA) | NaN | NaN | NaN | NaN |
 | R2SCAN (METAGGA) | NaN | NaN | NaN | NaN |
 | wB97M+D3 (Hybrid) | NaN | NaN | NaN | NaN |
 | Coupled Cluster | NaN | NaN | NaN | NaN |